In [ ]:
import json
import os
import re
import time
import urllib.parse
import urllib.request
from dotenv import load_dotenv
from exa_py import Exa
import geopandas as gpd
from shapely.geometry import shape
from google import genai
from google.genai import types

load_dotenv()

# Gemini with Google Search grounding via Vertex AI (GCP). Run once: gcloud auth application-default login
gemini_client = genai.Client(
    vertexai=True,
    project="miscellaneous-440005",
    location="us-central1",
)

# OSM Nominatim for suburb boundary polygons (single source for area)
NOMINATIM_URL = "https://nominatim.openstreetmap.org/search"
NOMINATIM_HEADERS = {"User-Agent": "nuvo-health-suburb-area/1.0 (contact via Nominatim usage policy)"}
# NSW: use GDA2020 / MGA zone 56 for accurate m² area
AREA_CRS = "EPSG:28356"
# ABS 2021 Suburbs and Localities (SAL) — layer 5 of General Community Profile Feature Server
ABS_SAL_FEATURESERVER = "https://services1.arcgis.com/vHnIGBHHqDR6y0CR/arcgis/rest/services/2021_ABS_General_Community_Profile/FeatureServer/5"
ABS_QUICKSTATS_BASE = "https://www.abs.gov.au/census/find-census-data/quickstats/2021"
exa = Exa(api_key=os.environ.get("EXA_API_KEY"))

CLINIC_COUNT_SCHEMA = {
    "type": "object",
    "properties": {
        "clinic_count": {
            "type": "integer",
            "description": "The total number of clinics only. Return just this number, no other text.",
        }
    },
    "required": ["clinic_count"],
    "additionalProperties": False,
}


def get_sal_code(suburb: str) -> str | None:
    """Fetch ABS 2021 SAL (Suburbs and Localities) code for a suburb from the ABS ArcGIS API.
    Returns the code (e.g. 'SAL10716' for Burwood) or None. Tries exact name then 'Suburb (NSW)'."""
    for name in (suburb.strip(), f"{suburb.strip()} (NSW)"):
        params = {
            "where": f"SAL_NAME_2021 = '{name.replace(chr(39), chr(39)+chr(39))}'",
            "outFields": "SAL_CODE_2021",
            "returnGeometry": "false",
            "f": "json",
        }
        url = ABS_SAL_FEATURESERVER + "/query?" + urllib.parse.urlencode(params)
        try:
            with urllib.request.urlopen(url, timeout=15) as resp:
                data = json.loads(resp.read().decode())
            for f in data.get("features", []):
                code = (f.get("attributes") or {}).get("SAL_CODE_2021")
                if code:
                    return code if code.startswith("SAL") else f"SAL{code}"
        except Exception:
            pass
    return None


def get_sal_codes(suburbs: list[str], checkpoint_path: str | None = None) -> dict[str, str | None]:
    """Get SAL codes for all suburbs. Returns {suburb_name: SAL_code or None}. Use checkpoint_path to save and resume."""
    result = {}
    if checkpoint_path and os.path.isfile(checkpoint_path):
        with open(checkpoint_path) as f:
            result = json.load(f)
    to_fetch = [s for s in suburbs if s not in result]
    for suburb in to_fetch:
        result[suburb] = get_sal_code(suburb)
        print(f"{suburb}: {result[suburb] or 'no SAL code'}")
        if checkpoint_path:
            with open(checkpoint_path, "w") as f:
                json.dump(result, f, indent=2)
    return result


def abs_quickstats_url(suburb: str, sal_code: str | None = None) -> str | None:
    """Build ABS 2021 Census QuickStats URL for a suburb. If sal_code is None, looks it up. Example: Burwood -> https://www.abs.gov.au/census/find-census-data/quickstats/2021/SAL10716"""
    code = sal_code or get_sal_code(suburb)
    if not code:
        return None
    return f"{ABS_QUICKSTATS_BASE}/{code}"


# Age fields for 65+ from ABS 2021 General Community Profile (layer 5)
ABS_AGE65_FIELDS = "Tot_P_P,Age_65_74_yr_P,Age_75_84_yr_P,Age_85ov_P"


def _fetch_abs_age_stats_for_suburb(suburb: str, sal_code: str | None) -> dict | None:
    """Query ABS Feature Server for one suburb; return attributes with Tot_P_P and 65+ bands or None."""
    attrs = None
    if sal_code:
        # Layer stores SAL_CODE_2021 as numeric e.g. "10716", not "SAL10716"
        code = str(sal_code).removeprefix("SAL") if str(sal_code).startswith("SAL") else str(sal_code)
        where = f"SAL_CODE_2021 = '{code}'"
    else:
        where = None
    for name in (suburb.strip(), f"{suburb.strip()} (NSW)") if not where else [None]:
        if name is not None:
            where = f"SAL_NAME_2021 = '{name.replace(chr(39), chr(39)+chr(39))}'"
        params = {
            "where": where,
            "outFields": ABS_AGE65_FIELDS,
            "returnGeometry": "false",
            "f": "json",
        }
        url = ABS_SAL_FEATURESERVER + "/query?" + urllib.parse.urlencode(params)
        try:
            with urllib.request.urlopen(url, timeout=15) as resp:
                data = json.loads(resp.read().decode())
            for f in data.get("features", []):
                a = f.get("attributes") or {}
                if a.get("Tot_P_P") is not None:
                    return a
        except Exception:
            pass
    return None


def get_pct_65_plus_abs(suburb: str, sal_code: str | None = None) -> float | None:
    """Percentage of population aged 65+ from ABS 2021 Census (SAL). Uses age bands 65–74, 75–84, 85+. Returns 0–100 or None."""
    attrs = _fetch_abs_age_stats_for_suburb(suburb, sal_code)
    if not attrs:
        return None
    total = attrs.get("Tot_P_P")
    if total is None or total == 0:
        return None
    age65 = (
        (attrs.get("Age_65_74_yr_P") or 0)
        + (attrs.get("Age_75_84_yr_P") or 0)
        + (attrs.get("Age_85ov_P") or 0)
    )
    return round(100.0 * age65 / total, 2)

def get_pct_65_plus_abs_batch(
    suburbs: list[str],
    sal_codes: dict[str, str | None] | None = None,
    checkpoint_path: str | None = None,
) -> dict[str, float | None]:
    """Get % population aged 65+ from ABS for many suburbs. Optional sal_codes dict and checkpoint_path to save/resume."""
    result = {}
    if checkpoint_path and os.path.isfile(checkpoint_path):
        with open(checkpoint_path) as f:
            result = json.load(f)
    sal_codes = sal_codes or {}
    to_fetch = [s for s in suburbs if s not in result]
    for suburb in to_fetch:
        code = sal_codes.get(suburb)
        pct = get_pct_65_plus_abs(suburb, sal_code=code)
        result[suburb] = pct
        print(f"{suburb}: {pct}% 65+" if pct is not None else f"{suburb}: no ABS data")
        if checkpoint_path:
            with open(checkpoint_path, "w") as f:
                json.dump(result, f, indent=2)
    return result


def get_population_abs(suburb: str, sal_code: str | None = None) -> int | None:
    """Total population from ABS 2021 Census (SAL). Same source as over-65%; uses Tot_P_P. Returns int or None."""
    attrs = _fetch_abs_age_stats_for_suburb(suburb, sal_code)
    if not attrs:
        return None
    total = attrs.get("Tot_P_P")
    if total is None:
        return None
    return int(total)


def get_population_abs_batch(
    suburbs: list[str],
    sal_codes: dict[str, str | None] | None = None,
    checkpoint_path: str | None = None,
) -> dict[str, int | None]:
    """Get total population from ABS for many suburbs. Same method as over-65%. Optional sal_codes and checkpoint_path."""
    result = {}
    if checkpoint_path and os.path.isfile(checkpoint_path):
        with open(checkpoint_path) as f:
            result = json.load(f)
    sal_codes = sal_codes or {}
    to_fetch = [s for s in suburbs if s not in result]
    for suburb in to_fetch:
        code = sal_codes.get(suburb)
        pop = get_population_abs(suburb, sal_code=code)
        result[suburb] = pop
        print(f"{suburb}: {pop}" if pop is not None else f"{suburb}: no ABS data")
        if checkpoint_path:
            with open(checkpoint_path, "w") as f:
                json.dump(result, f, indent=2)
    return result


# Income fields from ABS 2021 General Community Profile (layer 5, SAL)
ABS_INCOME_FIELDS = "Median_tot_hhd_inc_weekly,Median_tot_fam_inc_weekly,Median_tot_prsnl_inc_weekly"


def _fetch_abs_income_for_suburb(suburb: str, sal_code: str | None) -> dict | None:
    """Query ABS Feature Server (SAL layer) for one suburb; return attributes with the three income medians or None."""
    attrs = None
    if sal_code:
        code = str(sal_code).removeprefix("SAL") if str(sal_code).startswith("SAL") else str(sal_code)
        where = f"SAL_CODE_2021 = '{code}'"
    else:
        where = None
    for name in (suburb.strip(), f"{suburb.strip()} (NSW)") if not where else [None]:
        if name is not None:
            where = f"SAL_NAME_2021 = '{name.replace(chr(39), chr(39)+chr(39))}'"
        params = {
            "where": where,
            "outFields": ABS_INCOME_FIELDS,
            "returnGeometry": "false",
            "f": "json",
        }
        url = ABS_SAL_FEATURESERVER + "/query?" + urllib.parse.urlencode(params)
        try:
            with urllib.request.urlopen(url, timeout=15) as resp:
                data = json.loads(resp.read().decode())
            for f in data.get("features", []):
                a = f.get("attributes") or {}
                if any(a.get(k) is not None for k in ("Median_tot_hhd_inc_weekly", "Median_tot_fam_inc_weekly", "Median_tot_prsnl_inc_weekly")):
                    return a
        except Exception:
            pass
    return None


def get_income_abs(suburb: str, sal_code: str | None = None) -> dict | None:
    """Median household, family and personal weekly income from ABS 2021 (SAL). Returns dict with keys
    median_household_income_weekly, median_family_income_weekly, median_personal_income_weekly ($) or None."""
    attrs = _fetch_abs_income_for_suburb(suburb, sal_code)
    if not attrs:
        return None
    return {
        "median_household_income_weekly": attrs.get("Median_tot_hhd_inc_weekly") if attrs.get("Median_tot_hhd_inc_weekly") is not None else None,
        "median_family_income_weekly": attrs.get("Median_tot_fam_inc_weekly") if attrs.get("Median_tot_fam_inc_weekly") is not None else None,
        "median_personal_income_weekly": attrs.get("Median_tot_prsnl_inc_weekly") if attrs.get("Median_tot_prsnl_inc_weekly") is not None else None,
    }


def get_income_abs_batch(
    suburbs: list[str],
    sal_codes: dict[str, str | None] | None = None,
    checkpoint_path: str | None = None,
) -> dict[str, dict | None]:
    """Get median household/family/personal weekly income from ABS for many suburbs. Saves one JSON per suburb entry."""
    result = {}
    if checkpoint_path and os.path.isfile(checkpoint_path):
        with open(checkpoint_path) as f:
            result = json.load(f)
    sal_codes = sal_codes or {}
    to_fetch = [s for s in suburbs if s not in result]
    for suburb in to_fetch:
        code = sal_codes.get(suburb)
        row = get_income_abs(suburb, sal_code=code)
        result[suburb] = row
        print(f"{suburb}: {row}" if row else f"{suburb}: no ABS data")
        if checkpoint_path:
            with open(checkpoint_path, "w") as f:
                json.dump(result, f, indent=2)
    return result


def _polygon_area_km2(geojson_geom: dict) -> float | None:
    """Compute area in km² from a GeoJSON geometry (Polygon/MultiPolygon). Projects to NSW CRS for accurate m²."""
    try:
        geom = shape(geojson_geom)
        if geom.is_empty or geom.area == 0:
            return None
        gdf = gpd.GeoDataFrame(geometry=[geom], crs="EPSG:4326")
        gdf_proj = gdf.to_crs(AREA_CRS)
        area_m2 = float(gdf_proj.geometry.area.iloc[0])
        return area_m2 / 1_000_000.0
    except Exception:
        return None


def _get_suburb_boundary_osm(suburb: str) -> dict | None:
    """Fetch suburb boundary polygon from OSM Nominatim. Requests multiple results and
    returns the smallest polygon by area (suburb rather than council/LGA). Returns GeoJSON geometry dict or None."""
    params = {
        "city": suburb.strip(),
        "state": "New South Wales",
        "country": "Australia",
        "format": "json",
        "polygon_geojson": 1,
        "limit": 10,
        "countrycodes": "au",
        "featureType": "settlement",
    }
    req = urllib.request.Request(
        NOMINATIM_URL + "?" + urllib.parse.urlencode(params),
        headers=NOMINATIM_HEADERS,
    )
    try:
        with urllib.request.urlopen(req, timeout=15) as resp:
            data = json.loads(resp.read().decode())
        if not data or not isinstance(data, list):
            return None
        best_geojson = None
        best_area_km2 = float("inf")
        for item in data:
            geojson = item.get("geojson")
            if not geojson:
                continue
            kind = geojson.get("type")
            if kind not in ("Polygon", "MultiPolygon"):
                continue
            area = _polygon_area_km2(geojson)
            if area is not None and area < best_area_km2:
                best_area_km2 = area
                best_geojson = geojson
        return best_geojson
    except Exception:
        return None


def get_suburb_area(
    suburbs: list[str],
    checkpoint_path: str | None = None,
) -> dict[str, float | None]:
    """Fetch suburb area (km²) from OSM boundary polygon. Single source of truth.
    Uses Nominatim to get the actual boundary geometry, projects to metric CRS (NSW), and computes area.
    Returns {suburb_name: area_km2 or None}. Respects Nominatim 1 req/s; use checkpoint_path to resume."""
    result = {}
    if checkpoint_path and os.path.isfile(checkpoint_path):
        with open(checkpoint_path) as f:
            result = json.load(f)
    to_fetch = [s for s in suburbs if s not in result]
    for suburb in to_fetch:
        time.sleep(1.05)
        geom = _get_suburb_boundary_osm(suburb)
        if geom is None:
            result[suburb] = None
            print(f"{suburb}: no boundary polygon")
        else:
            area = _polygon_area_km2(geom)
            if area is None:
                result[suburb] = None
                print(f"{suburb}: could not compute area")
            else:
                result[suburb] = round(area, 4)
                print(f"{suburb}: {result[suburb]} km²")
        if checkpoint_path:
            with open(checkpoint_path, "w") as f:
                json.dump(result, f, indent=2)
    return result


CLINIC_COUNT_SYSTEM_PROMPT = (
    "Count only clinics (medical/GP/health clinics) within the Statistical Area Level 2 (SA2) boundary for the suburb — "
    "not the local government area (LGA) or council area. Use the same geographic boundary as ABS SA2. "
    "Return only the total count as an integer. Prefer official or authoritative sources where possible."
)


def get_clinic_count(
    suburbs: list[str],
    checkpoint_path: str | None = None,
) -> dict[str, int | None]:
    """Fetch total number of clinics in each suburb as SA2 in Sydney NSW.
    Returns {suburb_name: clinic_count or None}. Use checkpoint_path to save and avoid re-fetching (results can vary between runs)."""
    result = {}
    if checkpoint_path and os.path.isfile(checkpoint_path):
        with open(checkpoint_path) as f:
            result = json.load(f)
    to_fetch = [s for s in suburbs if s not in result]
    for suburb in to_fetch:
        query = (
            f"How many clinics (medical, GP, health clinics) are in the suburb {suburb}, Sydney NSW, "
            "defined as Statistical Area Level 2 (SA2)? Not LGA or council — SA2 only. Reply with only the total number."
        )
        try:
            response = exa.answer(
                query,
                text=True,
                output_schema=CLINIC_COUNT_SCHEMA,
                system_prompt=CLINIC_COUNT_SYSTEM_PROMPT,
            )
            ans = response.answer
            count = ans.get("clinic_count") if isinstance(ans, dict) else None
            result[suburb] = count
            print(f"{suburb}: {count if count is not None else 'N/A'} clinics")
        except Exception as e:
            result[suburb] = None
            print(f"{suburb}: error — {e}")
        if checkpoint_path:
            with open(checkpoint_path, "w") as f:
                json.dump(result, f, indent=2)
    return result


In [88]:
test_suburbs = [
  "Acacia Gardens",
  "Agnes Banks",
  "Airds",
  "Alexandria",
  "Alfords Point",
  "Allambie Heights",
  "Allawah",
  "Ambarvale",  
  "Angus",
  "Annandale",
  "Annangrove",
  "Arcadia",
  "Arncliffe",
]

In [32]:
# Total population from ABS 2021 (SAL), same method as over-65%. Saves to scraped_data/suburb_populations.json; re-run to resume.
with open("sydney_suburbs.json") as f:
    all_suburbs = json.load(f)

populations = get_population_abs_batch(all_suburbs, checkpoint_path="scraped_data/suburb_populations.json")

Abbotsbury: 4200
Abbotsford: 5431
Acacia Gardens: 3668
Agnes Banks: 996
Airds: 3265
Alexandria: 9649
Alfords Point: 3131
Allambie Heights: 7317
Allawah: 5351
Ambarvale: 7254
Angus: 384
Annandale: 9487
Annangrove: 1472
Arcadia: 1475
Arncliffe: 12023
Arndell Park: 6
Artarmon: 9417
Ashbury: 3353
Ashcroft: 3634
Ashfield: 23012
Asquith: 6160
Auburn: 39333
Austral: 6847
Avalon Beach: 10379
Badgerys Creek: 168
Balgowlah: 8068
Balgowlah Heights: 3546
Balmain: 10454
Balmain East: 1900
Bangor: 5536
Banksia: 3277
Banksmeadow: 508
Bankstown: 34933
Bankstown Aerodrome: 5
Barangaroo: 220
Barden Ridge: 4129
Bardia: 5747
Bardwell Park: 2320
Bardwell Valley: 2344
Bass Hill: 10230
Baulkham Hills: 37415
Bayview: 3807
Beacon Hill: 7814
Beaconsfield: 1172
Beaumont Hills: 9041
Beecroft: 10291
Belfield: 6555
Bella Vista: 8384
Bellevue Hill: 10590
Belmore: 13781
Belrose: 8726
Berala: 8757
Berkshire Park: 2149
Berowra: 4762
Berowra Heights: 5286
Berowra Waters: 169
Berrilee: 212
Beverley Park: 2646
Beverly Hil

In [27]:
# Google Maps Geocoding API — single source for suburb area (km² from viewport bounds).
get_suburb_area(test_suburbs)

Abbotsford: 1.0138 km²
Burwood: 2.4539 km²
Concord West: 2.7448 km²


{'Abbotsford': 1.0138, 'Burwood': 2.4539, 'Concord West': 2.7448}

In [29]:
# Full dataset: run get_suburb_area on all Sydney suburbs (Google Maps Geocoding).
# Saves to scraped_data/suburb_areas.json after each suburb; re-run to resume.
with open("sydney_suburbs.json") as f:
    all_suburbs = json.load(f)

areas = get_suburb_area(all_suburbs, checkpoint_path="scraped_data/suburb_areas.json")

Abbotsbury: 4.9792 km²
Abbotsford: 1.0138 km²
Acacia Gardens: 0.9838 km²
Agnes Banks: 16.1837 km²
Airds: 2.3654 km²
Alexandria: 3.5339 km²
Alfords Point: 3.03 km²
Allambie Heights: 6.5438 km²
Allawah: 0.5876 km²
Ambarvale: 2.755 km²
Angus: 7.598 km²
Annandale: 1.4997 km²
Annangrove: 10.1654 km²
Arcadia: 22.4328 km²
Arncliffe: 2.9469 km²
Arndell Park: 1.9687 km²
Artarmon: 2.5301 km²
Ashbury: 1.0486 km²
Ashcroft: 1.1564 km²
Ashfield: 3.3894 km²
Asquith: 3.4622 km²
Auburn: 8.5724 km²
Austral: 15.4664 km²
Avalon Beach: 5.3371 km²
Badgerys Creek: 21.4626 km²
Balgowlah: 1.9406 km²
Balgowlah Heights: 1.6624 km²
Balmain: 1.5489 km²
Balmain East: 0.2651 km²
Bangor: 4.1139 km²
Banksia: 0.9545 km²
Banksmeadow: 3.3274 km²
Bankstown: 6.4716 km²
Bankstown Aerodrome: 3.751 km²
Barangaroo: 0.2107 km²
Barden Ridge: 5.4679 km²
Bardia: 2.1333 km²
Bardwell Park: 0.916 km²
Bardwell Valley: 0.7337 km²
Bass Hill: 2.9709 km²
Baulkham Hills: 13.7269 km²
Bayview: 3.5703 km²
Beacon Hill: 2.7562 km²
Beaconsfield:

In [3]:
# Full list: % population aged 65+ from ABS 2021 (SAL). Saves to scraped_data/suburb_over_65_pct_abs.json after each suburb; re-run to resume.
with open("sydney_suburbs.json") as f:
    all_suburbs = json.load(f)

pct_65_plus = get_pct_65_plus_abs_batch(all_suburbs, checkpoint_path="scraped_data/suburb_over_65_pct.json")

Abbotsbury: 15.36% 65+
Abbotsford: 26.11% 65+
Acacia Gardens: 8.78% 65+
Agnes Banks: 13.96% 65+
Airds: 10.05% 65+
Alexandria: 6.17% 65+
Alfords Point: 16.16% 65+
Allambie Heights: 19.2% 65+
Allawah: 14.6% 65+
Ambarvale: 12.94% 65+
Angus: 16.41% 65+
Annandale: 13.61% 65+
Annangrove: 18.55% 65+
Arcadia: 22.1% 65+
Arncliffe: 12.67% 65+
Arndell Park: 0.0% 65+
Artarmon: 12.77% 65+
Ashbury: 19.21% 65+
Ashcroft: 11.64% 65+
Ashfield: 16.81% 65+
Asquith: 11.54% 65+
Auburn: 10.15% 65+
Austral: 9.77% 65+
Avalon Beach: 21.61% 65+
Badgerys Creek: 29.17% 65+
Balgowlah: 16.55% 65+
Balgowlah Heights: 15.12% 65+
Balmain: 16.31% 65+
Balmain East: 20.42% 65+
Bangor: 22.29% 65+
Banksia: 18.71% 65+
Banksmeadow: 5.12% 65+
Bankstown: 12.34% 65+
Bankstown Aerodrome: 0.0% 65+
Barangaroo: 8.64% 65+
Barden Ridge: 13.71% 65+
Bardia: 2.89% 65+
Bardwell Park: 18.45% 65+
Bardwell Valley: 19.24% 65+
Bass Hill: 15.11% 65+
Baulkham Hills: 16.17% 65+
Bayview: 40.64% 65+
Beacon Hill: 15.89% 65+
Beaconsfield: 5.46% 65+
Be

In [2]:
# All suburbs → one JSON file
with open("sydney_suburbs.json") as f:
    all_suburbs = json.load(f)
incomes = get_income_abs_batch(all_suburbs, checkpoint_path="scraped_data/suburb_income.json")
# Optional: save again to one file
with open("scraped_data/suburb_income.json", "w") as f:
    json.dump(incomes, f, indent=2)

Abbotsbury: {'median_household_income_weekly': 2621, 'median_family_income_weekly': 2684, 'median_personal_income_weekly': 846}
Abbotsford: {'median_household_income_weekly': 2263, 'median_family_income_weekly': 2988, 'median_personal_income_weekly': 1179}
Acacia Gardens: {'median_household_income_weekly': 2796, 'median_family_income_weekly': 2800, 'median_personal_income_weekly': 1055}
Agnes Banks: {'median_household_income_weekly': 2454, 'median_family_income_weekly': 2517, 'median_personal_income_weekly': 953}
Airds: {'median_household_income_weekly': 944, 'median_family_income_weekly': 1111, 'median_personal_income_weekly': 451}
Alexandria: {'median_household_income_weekly': 2654, 'median_family_income_weekly': 3600, 'median_personal_income_weekly': 1629}
Alfords Point: {'median_household_income_weekly': 2969, 'median_family_income_weekly': 2978, 'median_personal_income_weekly': 956}
Allambie Heights: {'median_household_income_weekly': 2890, 'median_family_income_weekly': 3298, 'me

In [3]:
import pandas as pd
from pathlib import Path

# Paths relative to project root
gp_data_dir = Path("gp_data")
out_path = Path("scraped_data") / "gp_clinics_all.csv"

# Collect and concatenate all CSVs in gp_data (same columns)
csv_files = sorted(gp_data_dir.glob("*.csv"))
dfs = [pd.read_csv(f) for f in csv_files]
combined = pd.concat(dfs, ignore_index=True)

# Union: drop duplicate rows (same clinic_name + suburb + number_of_gps)
combined = combined.drop_duplicates()

combined.to_csv(out_path, index=False)
print(f"Combined {len(csv_files)} CSVs → {len(combined)} rows → saved to {out_path}")

Combined 70 CSVs → 1322 rows → saved to scraped_data\gp_clinics_all.csv


In [8]:
import json
import numpy as np
import pandas as pd
from pathlib import Path

data_dir = Path("scraped_data")

# 1. Aggregate GP clinics: total_gp per suburb (sum of number_of_gps)
gp = pd.read_csv(data_dir / "gp_clinics_all.csv")
# Coerce: some rows have '>=6', 'unknown', blanks — avoids string concat bugs in groupby().sum()
gp["number_of_gps"] = pd.to_numeric(gp["number_of_gps"], errors="coerce").fillna(0)
total_gp = gp.groupby("suburb", as_index=False)["number_of_gps"].sum().rename(columns={"number_of_gps": "total_gp"})

# 2. Load JSON sources
with open(data_dir / "suburb_areas.json") as f:
    areas = json.load(f)
with open(data_dir / "suburb_populations.json") as f:
    populations = json.load(f)
with open(data_dir / "suburb_over_65_pct.json") as f:
    over_65_pct = json.load(f)
with open(data_dir / "suburb_income.json") as f:
    income = json.load(f)

# Manual imputation overrides (ABS 2021 QuickStats)
# Only fills values that are missing/blank in the scraped JSON.
manual_income_overrides = {
    "Silverwater": {
        "median_household_income_weekly": 1875,
        "median_family_income_weekly": 2055,
        "median_personal_income_weekly": 857,
    },
    "Enmore": {
        "median_household_income_weekly": 2309,
        "median_family_income_weekly": 3186,
        "median_personal_income_weekly": 1245,
    },
    "Darlington": {
        "median_household_income_weekly": 2392,
        "median_family_income_weekly": 3263,
        "median_personal_income_weekly": 780,
    },
    "Dural": {
        "median_household_income_weekly": 2489,
        "median_family_income_weekly": 2877,
        "median_personal_income_weekly": 909,
    },
    "Elderslie": {
        "median_household_income_weekly": 2150,
        "median_family_income_weekly": 2452,
        "median_personal_income_weekly": 955,
    },
    "Homebush West": {
        "median_household_income_weekly": 1952,
        "median_family_income_weekly": 2072,
        "median_personal_income_weekly": 860,
    },
    "Greendale": {
        "median_household_income_weekly": 1958,
        "median_family_income_weekly": 1812,
        "median_personal_income_weekly": 709,
    },
    "Kingswood": {
        "median_household_income_weekly": 1397,
        "median_family_income_weekly": 1755,
        "median_personal_income_weekly": 747,
    },
    "Lilli Pilli": {
        "median_household_income_weekly": 3625,
        "median_family_income_weekly": 3690,
        "median_personal_income_weekly": 1186,
    },
    "Long Point": {
        "median_household_income_weekly": 1775,
        "median_family_income_weekly": 1968,
        "median_personal_income_weekly": 900,
    },
    "Punchbowl": {
        "median_household_income_weekly": 1389,
        "median_family_income_weekly": 1483,
        "median_personal_income_weekly": 504,
    },
    "St Clair": {
        "median_household_income_weekly": 2067,
        "median_family_income_weekly": 2183,
        "median_personal_income_weekly": 849,
    },
    "Summer Hill": {
        "median_household_income_weekly": 2260,
        "median_family_income_weekly": 3076,
        "median_personal_income_weekly": 1317,
    },
    "The Rocks": {
        "median_household_income_weekly": 4374,
        "median_family_income_weekly": 4764,
        "median_personal_income_weekly": 1911,
    },
}

for suburb, overrides in manual_income_overrides.items():
    existing = income.get(suburb)
    if not isinstance(existing, dict):
        existing = {}
    for k, v in overrides.items():
        cur = existing.get(k)
        if cur in (None, "", "null"):
            existing[k] = v
    income[suburb] = existing

# Manual demographics (ABS 2021 QuickStats) — only fills missing JSON values
manual_demographics_overrides = {
    "Darlington": {"population": 2597, "over_65_pct": 5.43},
    "Dural": {"population": 7900, "over_65_pct": 22.11},
    "Elderslie": {"population": 7878, "over_65_pct": 13.05},
    "Enmore": {"population": 3871, "over_65_pct": 9.43},
    "Greendale": {"population": 314, "over_65_pct": 17.83},
    "Kingswood": {"population": 10633, "over_65_pct": 12.74},
    "Long Point": {"population": 173, "over_65_pct": 20.23},
    "Lilli Pilli": {"population": 1405, "over_65_pct": 19.15},
    "Punchbowl": {"population": 21384, "over_65_pct": 12.82},
    "Silverwater": {"population": 3600, "over_65_pct": 5.89},
    "St Clair": {"population": 19942, "over_65_pct": 12.71},
    "Summer Hill": {"population": 7288, "over_65_pct": 12.43},
    "The Rocks": {"population": 629, "over_65_pct": 18.76},
}
for suburb, dem in manual_demographics_overrides.items():
    if populations.get(suburb) in (None, "", "null"):
        populations[suburb] = dem["population"]
    if over_65_pct.get(suburb) in (None, "", "null"):
        over_65_pct[suburb] = dem["over_65_pct"]

# Suburbs permanently excluded from suburb-level outputs
EXCLUDED_SUBURBS = frozenset({
    "Bradfield",
    "Camellia",
    "Central Business District",
    "Currawong Beach",
    "East Gordon",
    "Flemington",
    "Green Square",
    "Huntingwood",
    "Kingswood Park",
    "Lansdowne",
    "Len Waters Estate",
    "Macarthur",
    "McCarrs Creek",
    "Nelson",
    "North St Ives",
    "Port Botany",
    "Royal National Park",
    "St James",
    "West Killara",
})

# 3. Build suburb-level tables from JSON
suburbs = sorted(
    (
        set(areas)
        | set(populations)
        | set(over_65_pct)
        | set(income)
        | set(total_gp["suburb"])
    )
    - EXCLUDED_SUBURBS
)
rows = []
for s in suburbs:
    r = {"suburb": s}
    r["total_gp"] = total_gp.loc[total_gp["suburb"] == s, "total_gp"].sum() or 0
    r["area"] = areas.get(s)
    r["population"] = populations.get(s)
    r["over_65_pct"] = over_65_pct.get(s)
    inc = income.get(s)
    if isinstance(inc, dict):
        r["median_household_income_weekly"] = inc.get("median_household_income_weekly")
        r["median_family_income_weekly"] = inc.get("median_family_income_weekly")
        r["median_personal_income_weekly"] = inc.get("median_personal_income_weekly")
    else:
        r["median_household_income_weekly"] = r["median_family_income_weekly"] = r["median_personal_income_weekly"] = None
    rows.append(r)

final = pd.DataFrame(rows)

# Cast numeric columns (avoid str vs int comparison)
for col in ["total_gp", "area", "population", "over_65_pct", "median_household_income_weekly", "median_family_income_weekly", "median_personal_income_weekly"]:
    final[col] = pd.to_numeric(final[col], errors="coerce")

# Keep all suburbs; total_gp = 0 where no clinics, nulls stay as empty in CSV
final_clean = final.copy()
final_clean["total_gp"] = final_clean["total_gp"].fillna(0)

# Column order
cols = [
    "suburb", "total_gp", "area", "population", "over_65_pct",
    "median_household_income_weekly", "median_family_income_weekly", "median_personal_income_weekly",
]
final_clean = final_clean[cols]

out_path = data_dir / "suburbs_final.csv"
final_clean.to_csv(out_path, index=False)
print(f"Saved {len(final_clean)} suburbs to {out_path}")

# EDA export: base columns + derived fields (patients/clinic, seniors, affordability, people/km²)
eda = final_clean.copy()
pop = eda["population"]
gp_n = eda["total_gp"]
eda["gp_density"] = pop / (gp_n + 1)
eda["seniors_count"] = pop * (eda["over_65_pct"] / 100.0)
hh = eda["median_household_income_weekly"]
pers = eda["median_personal_income_weekly"]
eda["affordability_index"] = np.where(
    (pers.notna()) & (pers != 0), hh / pers, np.nan
)
area_km2 = eda["area"]
eda["spatial_density"] = np.where(
    (area_km2.notna()) & (area_km2 > 0), pop / area_km2, np.nan
)
eda["gp_density"] = eda["gp_density"].round(4)
eda["seniors_count"] = eda["seniors_count"].round(2)
eda["affordability_index"] = eda["affordability_index"].round(6)
eda["spatial_density"] = eda["spatial_density"].round(4)

# Final opportunity score: w1*Norm(density) + w2*Norm(income) + w3*Norm(age); min-max [0,1] on all suburbs
def _opp_mm_high(s):
    v = pd.to_numeric(s, errors="coerce").astype(float)
    med = float(np.nanmedian(v.to_numpy()))
    v = v.fillna(med)
    lo, hi = float(v.min()), float(v.max())
    if hi == lo or not np.isfinite(lo) or not np.isfinite(hi):
        return pd.Series(0.5, index=s.index, dtype=float)
    return ((v - lo) / (hi - lo)).clip(0.0, 1.0)


_w1, _w2, _w3 = 1.0 / 3.0, 1.0 / 3.0, 1.0 / 3.0
_inc_s = eda["median_household_income_weekly"].replace(0, np.nan)
eda["opportunity_score"] = (
    _w1 * _opp_mm_high(eda["spatial_density"])
    + _w2 * _opp_mm_high(_inc_s)
    + _w3 * _opp_mm_high(eda["over_65_pct"])
).round(6)

eda_cols = cols + [
    "gp_density",
    "seniors_count",
    "affordability_index",
    "spatial_density",
    "opportunity_score",
]
eda = eda[eda_cols]
eda_path = data_dir / "eda_data.csv"
eda.to_csv(eda_path, index=False)
print(f"Saved {len(eda)} suburbs with EDA columns to {eda_path}")

Saved 699 suburbs to scraped_data\suburbs_final.csv


In [13]:
df = pd.read_csv("scraped_data/suburbs_final.csv")

df


,suburb,total_gp,area,population,over_65_pct,median_household_income_weekly,median_family_income_weekly,median_personal_income_weekly
0,Abbotsbury,0.0,4.9792,4200.0,15.36,2621.0,2684.0,846.0
1,Abbotsford,13.0,1.0138,5431.0,26.11,2263.0,2988.0,1179.0
2,Acacia Gardens,0.0,0.9838,3668.0,8.78,2796.0,2800.0,1055.0
3,Agnes Banks,0.0,16.1837,996.0,13.96,2454.0,2517.0,953.0
4,Airds,4.0,2.3654,3265.0,10.05,944.0,1111.0,451.0
...,...,...,...,...,...,...,...,...
694,Yarramundi,0.0,32.3404,829.0,14.11,2829.0,2814.0,966.0
695,Yarrawarrah,2.0,1.2166,2775.0,19.53,2724.0,2856.0,1020.0
696,Yennora,0.0,2.7411,1675.0,25.85,900.0,1299.0,448.0
697,Yowie Bay,0.0,1.8020,3053.0,21.85,2959.0,3336.0,1085.0


In [ ]:
import pandas as pd
df = pd.read_csv("scraped_data/suburbs_final.csv")
null_rows = df[df.isna().any(axis=1)]
print(f"Rows with at least one null: {len(null_rows)}")
null_rows

Rows with at least one null: 31


,suburb,total_gp,area,population,over_65_pct,median_household_income_weekly,median_family_income_weekly,median_personal_income_weekly
83,Bradfield,0.0,17.4037,NaN,NaN,NaN,NaN,NaN
104,Camellia,0.0,0.7405,0.0,NaN,1125.0,0.0,1125.0
133,Central Business District,0.0,NaN,NaN,NaN,NaN,NaN,NaN
180,Currawong Beach,0.0,0.1957,NaN,NaN,NaN,NaN,NaN
185,Darlington,10.0,0.3837,NaN,NaN,NaN,NaN,NaN
205,Dural,571010.0,34.0119,NaN,NaN,NaN,NaN,NaN
208,East Gordon,0.0,NaN,NaN,NaN,NaN,NaN,NaN
223,Elderslie,18.0,5.1267,NaN,NaN,NaN,NaN,NaN
234,Enmore,53.0,0.4243,NaN,NaN,NaN,NaN,NaN
248,Flemington,221.0,NaN,7025.0,12.0,1525.0,2191.0,890.0


In [ ]:
import pandas as pd
from pathlib import Path

p = Path("scraped_data/suburbs_final.csv")
df = pd.read_csv(p)

# Normalize empty strings to NA
for c in df.columns:
    if df[c].dtype == object:
        df[c] = df[c].replace(r"^\s*$", pd.NA, regex=True)

print("=== suburbs_final.csv quality report ===")
print(f"Rows: {len(df)}  Columns: {list(df.columns)}")
print()
print("--- Missing count per column ---")
for c in df.columns:
    n = df[c].isna().sum()
    pct = 100 * n / len(df)
    print(f"  {c}: {int(n)} ({pct:.1f}%)")

inc_cols = [
    "median_household_income_weekly",
    "median_family_income_weekly",
    "median_personal_income_weekly",
]
all_inc_missing = df[inc_cols].isna().all(axis=1)
print()
print(f"--- All 3 median incomes missing: {all_inc_missing.sum()} suburbs ---")
if all_inc_missing.any():
    print(df.loc[all_inc_missing, "suburb"].head(40).to_string(index=False))
    if all_inc_missing.sum() > 40:
        print(f"  ... and {all_inc_missing.sum() - 40} more")

pop_missing = df["population"].isna()
print()
print(f"--- population missing: {pop_missing.sum()} ---")

area_missing = df["area"].isna()
print(f"--- area missing: {area_missing.sum()} ---")

o65_missing = df["over_65_pct"].isna()
print(f"--- over_65_pct missing: {o65_missing.sum()} ---")

partial = df[inc_cols].isna().any(axis=1) & ~all_inc_missing
print()
print(f"--- Partial income (1-2 of 3 missing): {partial.sum()} suburbs ---")
if partial.any():
    s = df.loc[partial, ["suburb"] + inc_cols]
    print(s.to_string(index=False))

has_gp = df["total_gp"].fillna(0) > 0
demo_missing = df["population"].isna() | df["over_65_pct"].isna()
print()
print(
    f"--- Has clinics (total_gp>0) but population OR over_65 missing: {(has_gp & demo_missing).sum()} ---"
)
sub = df.loc[has_gp & demo_missing, ["suburb", "total_gp", "population", "over_65_pct"]]
if len(sub):
    print(sub.to_string(index=False))

core = ["area", "population", "over_65_pct"] + inc_cols
ghost = (df["total_gp"].fillna(0) == 0) & df[core].isna().all(axis=1)
print()
print(
    f"--- total_gp=0 AND all of area/pop/over_65/3 incomes missing: {ghost.sum()} ---"
)
if ghost.any():
    print(df.loc[ghost, "suburb"].head(25).to_string(index=False))


In [21]:
# Data quality: missing values in suburbs_final.csv (Python only — no PowerShell here)
import pandas as pd
from pathlib import Path

p = Path("scraped_data/suburbs_final.csv")
df = pd.read_csv(p)

# Normalize empty strings to NA
for c in df.columns:
    if df[c].dtype == object:
        df[c] = df[c].replace(r"^\s*$", pd.NA, regex=True)

print("=== suburbs_final.csv quality report ===")
print(f"Rows: {len(df)}  Columns: {list(df.columns)}")
print()
print("--- Missing count per column ---")
for c in df.columns:
    n = df[c].isna().sum()
    pct = 100 * n / len(df)
    print(f"  {c}: {int(n)} ({pct:.1f}%)")

inc_cols = [
    "median_household_income_weekly",
    "median_family_income_weekly",
    "median_personal_income_weekly",
]
all_inc_missing = df[inc_cols].isna().all(axis=1)
any_missing = df.isna().any(axis=1)
print()
print(f"--- Rows with ANY missing value: {any_missing.sum()} ---")
print(df.loc[any_missing].sort_values("suburb").to_string(index=False))
print()
print(f"--- All 3 median incomes missing: {all_inc_missing.sum()} suburbs ---")
if all_inc_missing.any():
    print(df.loc[all_inc_missing, "suburb"].head(40).to_string(index=False))
    if all_inc_missing.sum() > 40:
        print(f"  ... and {all_inc_missing.sum() - 40} more")

pop_missing = df["population"].isna()
print()
print(f"--- population missing: {pop_missing.sum()} ---")

area_missing = df["area"].isna()
print(f"--- area missing: {area_missing.sum()} ---")

o65_missing = df["over_65_pct"].isna()
print(f"--- over_65_pct missing: {o65_missing.sum()} ---")

partial = df[inc_cols].isna().any(axis=1) & ~all_inc_missing
print()
print(f"--- Partial income (1-2 of 3 missing): {partial.sum()} suburbs ---")
if partial.any():
    s = df.loc[partial, ["suburb"] + inc_cols]
    print(s.to_string(index=False))

has_gp = df["total_gp"].fillna(0) > 0
demo_missing = df["population"].isna() | df["over_65_pct"].isna()
print()
print(
    f"--- Has clinics (total_gp>0) but population OR over_65 missing: {(has_gp & demo_missing).sum()} ---"
)
sub = df.loc[has_gp & demo_missing, ["suburb", "total_gp", "population", "over_65_pct"]]
if len(sub):
    print(sub.to_string(index=False))

core = ["area", "population", "over_65_pct"] + inc_cols
ghost = (df["total_gp"].fillna(0) == 0) & df[core].isna().all(axis=1)
print()
print(
    f"--- total_gp=0 AND all of area/pop/over_65/3 incomes missing: {ghost.sum()} ---"
)
if ghost.any():
    print(df.loc[ghost, "suburb"].head(25).to_string(index=False))


=== suburbs_final.csv quality report ===
Rows: 685  Columns: ['suburb', 'total_gp', 'area', 'population', 'over_65_pct', 'median_household_income_weekly', 'median_family_income_weekly', 'median_personal_income_weekly']

--- Missing count per column ---
  suburb: 0 (0.0%)
  total_gp: 0 (0.0%)
  area: 1 (0.1%)
  population: 2 (0.3%)
  over_65_pct: 3 (0.4%)
  median_household_income_weekly: 1 (0.1%)
  median_family_income_weekly: 1 (0.1%)
  median_personal_income_weekly: 1 (0.1%)

--- Rows with ANY missing value: 4 ---
             suburb  total_gp   area  population  over_65_pct  median_household_income_weekly  median_family_income_weekly  median_personal_income_weekly
      McCarrs Creek       0.0 0.1204         NaN          NaN                             NaN                          NaN                            NaN
             Nelson       0.0 4.9550         NaN          NaN                          2399.0                       2416.0                          963.0
        Port Bot